# CAMUS — DRL Boundary Refinement

One notebook, three agents. Switch which agent runs by changing the YAML path in Cell 1:

| Agent | Config | Action space |
|---|---|---|
| **DDQN**   | `configs/camus_dqn.yaml`     | 7 discrete |
| **Dueling** | `configs/camus_dueling.yaml` | 7 discrete |
| **DDPG**   | `configs/camus_ddpg.yaml`    | 2D continuous |

Also pick which **structure** to refine via `target_class` in the YAML (1=LV_endo, 2=LV_epi, 3=LA). One run = one structure.

## Kaggle setup
1. Attach: **CAMUS dataset** + **iteris-pkg** + **camus-baseline-outputs** (the trained U-Net checkpoint)
2. Accelerator: **GPU T4 x2**, Persistence: **Files only**, Internet: **On**
3. Save Version → Save & Run All (Commit)

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'
subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Setup complete. Package at: {PKG_ROOT}')

## 1 · Load DRL config + baseline config
Edit `CFG_NAME` to pick the agent.

In [ ]:
CFG_NAME = 'camus_dqn.yaml'      # ← switch to camus_dueling.yaml or camus_ddpg.yaml

from iteris.config import load_config, load_drl_config
from iteris.utils  import get_device

cfg          = load_drl_config(str(PKG_ROOT / 'configs' / CFG_NAME))
baseline_cfg = load_config(    str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

# Locate CAMUS data root + baseline checkpoint regardless of mount path.
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])

ckpt_cands = [p for p in Path('/kaggle/input').rglob('camus_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])

cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({baseline_cfg["class_names"][cfg["target_class"]]})')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks
Runs the trained U-Net over the whole dataset once and caches predictions.
Avoids 50k+ wasted forward passes during DRL training.

In [ ]:
from iteris.warm_start import precompute_init_masks

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')
import numpy as np
init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val: mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · Run DRL training

In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']
best_dice = result['best_dice']

print(f'\n✓ Training complete. Best val final-Dice: {best_dice:.4f}')
print(f'  Checkpoint: {result["checkpoint"]}')

## 4 · Learning curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', linestyle='--', alpha=0.6)
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})')
ax1.set(xlabel='Training step', ylabel='Mean Val Dice', title='Refinement learning curve')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['step'], history['delta_dice_mean'], color='C2')
ax2.axhline(0, color='k', lw=0.5)
ax2.set(xlabel='Training step', ylabel='Δ Dice (final − init)',
        title=f'Refinement gain ({cfg["agent_type"]} c{cfg["target_class"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f'CAMUS {cfg["agent_type"]} — class {cfg["target_class"]} ({baseline_cfg["class_names"][cfg["target_class"]]})')
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_curves.png', dpi=150)
plt.show()

## 5 · Test-set evaluation

In [ ]:
from iteris.drl_training import evaluate_agent

test_metrics = evaluate_agent(agent, test_samples, env_kwargs={
    'action_type': agent.action_type,
    'max_steps':   cfg.get('max_steps', 20),
    'shift_px':    cfg.get('shift_px', 2),
    'sdt_clip':    cfg.get('sdt_clip', 20.0),
    'reward_clip': cfg.get('reward_clip', 1.0),
    'cont_action_scale': cfg.get('cont_action_scale', 0.04),
})
import json
print(json.dumps(test_metrics, indent=2))

with open(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_test_results.json', 'w') as f:
    json.dump({**test_metrics, 'agent_type': cfg['agent_type'], 'target_class': cfg['target_class'], 'best_val_dice': best_dice}, f, indent=2)
print('Saved test results JSON.')

## 6 · Episode-replay helper
Collects per-step masks during greedy rollouts. Used by all visualisations below.

In [ ]:
from iteris.env import SegmentationEnv

def replay_episode(agent, sample, env_kwargs):
    """Run greedy rollout, return (mask trajectory, dice history, hd95 history, actions)."""
    env = SegmentationEnv(sample['image'], sample['gt_mask'], sample['init_mask'], **env_kwargs)
    state = env.reset()
    masks   = [env.mask.copy()]
    actions = []
    while True:
        if agent.action_type == 'discrete':
            a = agent.select_action(state, epsilon=0.0)
        else:
            a = agent.select_action(state, explore=False)
        actions.append(a)
        state, _, done, _ = env.step(a)
        masks.append(env.mask.copy())
        if done:
            break
    return masks, list(env.dice_history), list(env.hd95_history), actions

ENV_KW = dict(
    action_type=agent.action_type,
    max_steps=cfg.get('max_steps', 20),
    shift_px=cfg.get('shift_px', 2),
    sdt_clip=cfg.get('sdt_clip', 20.0),
    reward_clip=cfg.get('reward_clip', 1.0),
    cont_action_scale=cfg.get('cont_action_scale', 0.04),
)
print('Helper ready.')

## 7 · Before/After comparison grid
Best / median / worst Δ-Dice samples side-by-side. Reveals if "improvement" is consistent or driven by lucky outliers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from iteris.env import hd95_px

# Replay every test sample once, sort by gain to pick stratified examples
gains = []
for s in test_samples:
    masks, dices, _, _ = replay_episode(agent, s, ENV_KW)
    gains.append((dices[-1] - dices[0], s, masks[-1], dices))

gains.sort(key=lambda x: x[0])
picks = [gains[-1], gains[len(gains)//2], gains[0]]
picks_labels = ['BEST gain', 'MEDIAN gain', 'WORST gain']

CLASS_COLOR = baseline_cfg['class_colors'][cfg['target_class']]
cmap_single = plt.cm.colors.ListedColormap([CLASS_COLOR])

fig, axes = plt.subplots(len(picks), 4, figsize=(16, 4*len(picks)))
for row, ((gain, s, final_mask, dices), label) in enumerate(zip(picks, picks_labels)):
    img    = s['image']
    init   = s['init_mask']
    gt     = s['gt_mask']
    init_d, final_d = dices[0], dices[-1]
    init_h  = hd95_px(init, gt)
    final_h = hd95_px(final_mask, gt)

    cells = [
        ('Input',                   None,       None,    None),
        ('U-Net init',              init,       init_d,  init_h),
        (f'{cfg["agent_type"]} refined', final_mask, final_d, final_h),
        ('Ground Truth',            gt,         1.0,     0.0),
    ]
    for col, (title, mask, dice, hd) in enumerate(cells):
        ax = axes[row, col]
        ax.imshow(img, cmap='gray')
        if mask is not None:
            ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=cmap_single, alpha=0.5)
        if dice is not None:
            ax.set_title(f'{title}\nDice {dice:.3f}  HD95 {hd:.1f}px', fontsize=10)
        else:
            ax.set_title(f'{title}\n[{label}]  ΔDice {gain:+.4f}', fontsize=10)
        ax.axis('off')

cls_name = baseline_cfg['class_names'][cfg['target_class']]
plt.suptitle(f'CAMUS {cfg["agent_type"]} — {cls_name} (class {cfg["target_class"]})', fontsize=13)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_comparison.png', dpi=150)
plt.show()

## 8 · Iteration playback
One sample, all 20 steps. **This is the differentiator** (per CONTEXT.md §4 — feeds the UI's Iteration Playback page). Green contour = GT boundary.

In [ ]:
from scipy import ndimage

DISCRETE_NAMES = ['dilate', 'erode', '↑', '↓', '←', '→', 'no-op']

# Use the BEST-gain sample for the playback (most informative trajectory)
best_sample = picks[0][1]
masks, dices, hd95s, actions = replay_episode(agent, best_sample, ENV_KW)

n_steps = len(masks)
ncols = 5
nrows = int(np.ceil(n_steps / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
axes = axes.flatten()

gt_edge = best_sample['gt_mask'] ^ ndimage.binary_erosion(best_sample['gt_mask'], iterations=1)

for i, (mask, dice, hd) in enumerate(zip(masks, dices, hd95s)):
    ax = axes[i]
    ax.imshow(best_sample['image'], cmap='gray')
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap=cmap_single, alpha=0.55)
    ax.imshow(np.ma.masked_where(~gt_edge.astype(bool), gt_edge), cmap='Greens', alpha=0.8)

    if i == 0:
        title = f't=0 (init)\nDice {dice:.3f}'
    else:
        a = actions[i-1]
        if agent.action_type == 'discrete':
            a_str = DISCRETE_NAMES[int(a)]
        else:
            a_str = f'({a[0]*256:+.1f},{a[1]*256:+.1f})px'
        title = f't={i}  {a_str}\nDice {dice:.3f}  Δ {dice-dices[i-1]:+.4f}'
    ax.set_title(title, fontsize=8)
    ax.axis('off')

for j in range(n_steps, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'{cfg["agent_type"]} iteration playback — {best_sample.get("patient","?")} '
             f'{best_sample.get("view","")} {best_sample.get("phase","")}  '
             f'(green = GT boundary)', fontsize=12)
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_playback.png', dpi=150)
plt.show()

## 9 · Trajectory curves + action distribution
Diagnoses agent behaviour: monotonic improvement vs. oscillation, action diversity vs. degenerate policy.

In [ ]:
n_replay = min(12, len(test_samples))
replays = [replay_episode(agent, test_samples[i], ENV_KW) for i in range(n_replay)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 — Dice trajectories
ax = axes[0]
for masks, dices, _, _ in replays:
    ax.plot(dices, alpha=0.5, lw=1)
max_len = max(len(d) for _, d, _, _ in replays)
padded = np.stack([np.pad(d, (0, max_len-len(d)), constant_values=d[-1])
                   for _, d, _, _ in replays])
ax.plot(padded.mean(axis=0), color='black', lw=2.5, label='mean')
ax.set(xlabel='Step', ylabel='Dice', title='Per-sample Dice trajectory')
ax.legend(); ax.grid(alpha=0.3)

# Panel 2 — HD95 trajectories
ax = axes[1]
for _, _, hd, _ in replays:
    valid = [v for v in hd if not np.isnan(v)]
    ax.plot(valid, alpha=0.5, lw=1)
ax.set(xlabel='Step', ylabel='HD95 (px)', title='Per-sample HD95 trajectory')
ax.grid(alpha=0.3)

# Panel 3 — Action distribution
ax = axes[2]
all_actions = [a for _, _, _, acts in replays for a in acts]
if agent.action_type == 'discrete':
    counts = np.bincount(all_actions, minlength=7)
    ax.bar(range(7), counts / counts.sum())
    ax.set_xticks(range(7))
    ax.set_xticklabels(DISCRETE_NAMES, rotation=30)
    ax.set(ylabel='Frequency', title='Action distribution')
    ax.grid(alpha=0.3, axis='y')
else:
    arr = np.array(all_actions)
    ax.scatter(arr[:, 1]*256, arr[:, 0]*256, alpha=0.5, s=20)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    sc = cfg.get('cont_action_scale', 0.04) * 256
    ax.set_xlim(-sc*1.1, sc*1.1); ax.set_ylim(-sc*1.1, sc*1.1)
    ax.set(xlabel='dx (px)', ylabel='dy (px)', title='Continuous action distribution')
    ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.suptitle(f'{cfg["agent_type"]} behaviour analysis (class {cfg["target_class"]})')
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_behaviour.png', dpi=150)
plt.show()

# Summary print
print(f'\n── {cfg["agent_type"]} c{cfg["target_class"]} behaviour summary ──')
init_dices  = [d[0]  for _, d, _, _ in replays]
final_dices = [d[-1] for _, d, _, _ in replays]
print(f'  Avg init Dice  : {np.mean(init_dices):.4f}')
print(f'  Avg final Dice : {np.mean(final_dices):.4f}')
print(f'  Avg Δ Dice     : {np.mean([f-i for f,i in zip(final_dices, init_dices)]):+.4f}')
print(f'  Avg episode length: {np.mean([len(d)-1 for _, d, _, _ in replays]):.1f} steps')
if agent.action_type == 'discrete':
    most_used = DISCRETE_NAMES[np.argmax(counts)]
    print(f'  Most-used action: {most_used} ({counts.max()/counts.sum()*100:.0f}%)')